In [1]:
%load_ext sql

In [2]:
%sql sqlite:///northwind.db

Connecting to 'sqlite:///northwind.db'

## Query 1: Top 10 Customers by Total Revenue
Business Question: Who are our most valuable customers?

In [3]:
%%sql
SELECT * FROM Customers LIMIT 5;

Running query in 'sqlite:///northwind.db'

CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,Western Europe,12209,Germany,030-0074321,030-0076545
ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,Central America,05021,Mexico,(5) 555-4729,(5) 555-3745
ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,Central America,05023,Mexico,(5) 555-3932,None
AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,British Isles,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,Northern Europe,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


## Query 2: Top 10 Countries by Total Orders
This query identifies the top 10 countries by calculating 
their total number of orders placed.

# Insight
The top 3 customers by revenue are IT ($9.7M), B's Beverages ($6.2M), and Hungry Coyote Import Store ($5.7M). 
These accounts represent a significant concentration of revenue, suggesting the business should prioritize retention strategies 
and dedicated account management for these clients.

In [4]:
%%sql
SELECT 
    c.CompanyName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY c.CompanyName
ORDER BY TotalRevenue DESC
LIMIT 10;

Running query in 'sqlite:///northwind.db'

CompanyName,TotalRevenue
IT,9745371.29
B's Beverages,6154115.34
Hungry Coyote Import Store,5698023.67
Rancho grande,5559110.08
Gourmet Lanchonetes,5552309.8
Ana Trujillo Emparedados y helados,5534356.65
Ricardo Adocicados,5524517.31
Folies gourmandes,5505502.85
Let's Stop N Shop,5462198.02
LILA-Supermercado,5437438.34


## Query 2: Top 10 Countries by Total Orders
This query identifies the top 10 countries by calculating 
their total number of orders placed.

# Insight 
"The top countries by order volume are the USA (2,280), France (1,909), and Germany (1,895), all of which are major developed economies with strong 
purchasing power. Emerging markets such as Venezuela (726), Canada (520), and Italy (519) show lower order volumes but represent potential growth opportunities. The business should consider targeted marketing and localized strategies in underpenetratedmarkets to diversify revenue beyond its heavy reliance on the USA."

In [5]:
%%sql
SELECT 
    c.Country,
    COUNT(o.OrderID) AS TotalOrders
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.Country
ORDER BY TotalOrders DESC
LIMIT 10;

Running query in 'sqlite:///northwind.db'

Country,TotalOrders
USA,2280
France,1909
Germany,1895
Brazil,1659
UK,1234
Mexico,898
Spain,849
Venezuela,726
Canada,520
Italy,519


## Query 3: Top 10 Products by Total Revenue
This query identifies our top performing products by calculating 
total revenue after discounts across all orders.

# Insight
Côte de Blaye dominates product revenue at $53.2M, more than double the second highest product. This heavy reliance on a single product represents a business risk — if supply, pricing, or demand shifts, overall revenue could be significantly impacted. The business should investigate 
what drives Côte de Blaye's success and consider strategies to grow revenue from underperforming products.

In [6]:
%%sql
SELECT 
    p.ProductName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Products p
JOIN [Order Details] od ON p.ProductID = od.ProductID
GROUP BY p.ProductName
ORDER BY TotalRevenue DESC
LIMIT 10;

Running query in 'sqlite:///northwind.db'

ProductName,TotalRevenue
Côte de Blaye,53265895.23
Thüringer Rostbratwurst,24623469.23
Mishi Kobe Niku,19423037.5
Sir Rodney's Marmalade,16653807.36
Carnarvon Tigers,12604671.88
Raclette Courdavault,11216410.7
Manjimup Dried Apples,10664768.65
Tarte au sucre,9952936.07
Ipoh Coffee,9333374.7
Rössle Sauerkraut,9252765.44


## Query 4: Products Running Low on Stock
This query identifies products where current stock levels have 
fallen at or below their reorder threshold, flagging items that 
require urgent restocking to avoid lost sales.

## Insight 
Several products are critically out of stock with no reorders placed, most notably Thüringer Rostbratwurst — the company's second highest revenue product at $24.6M. The stockout pattern appears concentrated in meat products, which have shorter shelf lives and require tighter inventory management. The business should treat meat product restocking as a high priority, as current stockouts likely represent significant lost revenue.



In [7]:
%%sql
SELECT 
    p.ProductName,
    p.UnitsInStock,
    p.ReorderLevel,
    p.UnitsOnOrder
FROM Products p
WHERE p.UnitsInStock <= p.ReorderLevel
ORDER BY p.UnitsInStock ASC;

Running query in 'sqlite:///northwind.db'

ProductName,UnitsInStock,ReorderLevel,UnitsOnOrder
Chef Anton's Gumbo Mix,0,0,0
Alice Mutton,0,0,0
Thüringer Rostbratwurst,0,0,0
Gorgonzola Telino,0,20,70
Perth Pasties,0,0,0
Sir Rodney's Scones,3,5,40
Louisiana Hot Spiced Okra,4,20,100
Longlife Tofu,4,5,20
Rogede sild,5,15,70
Scottish Longbreads,6,15,10


## Query 5: Monthly Sales Trends Over Time
This query calculates total revenue per month after discounts, 
using date functions to group orders chronologically and identify 
sales patterns over time.

# Insight
2013 monthly revenue shows two distinct peak periods — July and September — followed by a building trend toward December. This suggests the business experiences both a mid year surge and a strong end of year climb, with softer periods in between. The business should align inventory and staffing around these three peak windows.




In [8]:
%%sql
SELECT 
    STRFTIME('%Y-%m', o.OrderDate) AS Month,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Orders o
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY Month
ORDER BY Month ASC;

Running query in 'sqlite:///northwind.db'

Month,TotalRevenue
2012-07,2066219.4
2012-08,3556875.79
2012-09,3440144.98
2012-10,3201529.96
2012-11,2980494.74
2012-12,3577936.85
2013-01,3075418.29
2013-02,2964192.86
2013-03,3471361.21
2013-04,3262893.52


## Query 6: Sales Performance by Employee
This query evaluates individual sales performance by calculating 
total revenue generated per employee after discounts, identifying 
top and bottom performers across the sales team.

# Insight
"Sales revenue is remarkably evenly distributed across all 9 employees, with less than 7% separating top performer Margaret Peacock ($51.5M) from bottom ranked Andrew Fuller ($48.3M). This suggests a structured order assignment system rather than individual sales hunting. The business should investigate whether performance based incentives could drive growth, or whether the current balanced distribution is intentional."

In [9]:
%%sql
SELECT 
    e.FirstName || ' ' || e.LastName AS EmployeeName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Employees e
JOIN Orders o ON e.EmployeeID = o.EmployeeID
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY e.EmployeeID
ORDER BY TotalRevenue DESC
LIMIT 10;

Running query in 'sqlite:///northwind.db'

EmployeeName,TotalRevenue
Margaret Peacock,51488395.2
Steven Buchanan,51386459.1
Janet Leverling,50445573.76
Nancy Davolio,49659423.23
Robert King,49651899.3
Laura Callahan,49281136.81
Michael Suyama,49139966.56
Anne Dodsworth,49019678.44
Andrew Fuller,48314100.77


## Query 7: Shipper Delivery Speed Comparison
This query calculates the average number of days each shipper 
takes to deliver orders, using date arithmetic to measure the 
time between order date and shipped date.

## Insight
All three shippers perform virtually identically, delivering orders in roughly 7.8 days on average with less than 3 hours separating fastest from slowest. Speed alone cannot justify choosing one shipper over another. The business should evaluate shippers on additional factors such as cost per shipment, damage rates, and customer satisfaction to make a more informed logistics decision. Speedy Express, despite its name, is ironically the slowest of the three.

In [10]:
%%sql
SELECT 
    s.CompanyName AS Shipper,
    ROUND(AVG(JULIANDAY(o.ShippedDate) - JULIANDAY(o.OrderDate)), 2) AS AvgDaysToShip
FROM Shippers s
JOIN Orders o ON s.ShipperID = o.ShipVia
WHERE o.ShippedDate IS NOT NULL
GROUP BY Shipper
ORDER BY AvgDaysToShip ASC;

Running query in 'sqlite:///northwind.db'

Shipper,AvgDaysToShip
United Package,7.78
Federal Shipping,7.82
Speedy Express,7.92


## Query 8: Average Order Value by Country
This query calculates the average revenue per order for each 
country after discounts, identifying which markets generate 
the highest value orders.

## Insight
While Belgium leads average order value at $748.19, the more striking finding is that Argentina and Venezuela rank 2nd and 3rd despite being developing economies with significantly lower purchasing power than other countries on the list. This contradicts expected spending patterns and warrants investigation — possible explanations include data entry errors, bulk ordering by a single large client, or currency conversion anomalies. Cross referencing with Query 2 reveals that the USA dominates total order volume but doesn't appear in the top 10 for average order value, suggesting American customers place frequent smaller orders while certain markets place fewer but larger ones. The business should investigate anomalous markets while considering strategies to increase average order value in high volume countries like the USA.



In [11]:
%%sql
SELECT 
    c.Country,
    ROUND(AVG(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS AvgOrderValue
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY c.Country
ORDER BY AvgOrderValue DESC
LIMIT 10;

Running query in 'sqlite:///northwind.db'

Country,AvgOrderValue
Belgium,748.19
Argentina,742.84
Venezuela,740.28
Switzerland,738.99
Germany,738.0
Sweden,737.54
UK,737.36
Finland,736.98
Brazil,736.78
Austria,736.73


In [12]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('northwind.db')

# Run Query 1
query = """
SELECT 
    c.CompanyName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY c.CompanyName
ORDER BY TotalRevenue DESC
LIMIT 10;
"""

# Save to CSV
df = pd.read_sql_query(query, conn)
df.to_csv('query1_top_customers.csv', index=False)
print("Exported successfully!")

Exported successfully!


In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('northwind.db')

# Query 2: Top Countries by Total Orders
query2 = """
SELECT 
    c.Country,
    COUNT(o.OrderID) AS TotalOrders
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.Country
ORDER BY TotalOrders DESC
LIMIT 10;
"""

# Query 3: Top Products by Total Revenue
query3 = """
SELECT 
    p.ProductName,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Products p
JOIN [Order Details] od ON p.ProductID = od.ProductID
GROUP BY p.ProductName
ORDER BY TotalRevenue DESC
LIMIT 10;
"""

# Query 5: Monthly Sales Trends
query5 = """
SELECT 
    STRFTIME('%Y-%m', o.OrderDate) AS Month,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalRevenue
FROM Orders o
JOIN [Order Details] od ON o.OrderID = od.OrderID
GROUP BY Month
ORDER BY Month ASC;
"""

# Export all to CSV
pd.read_sql_query(query2, conn).to_csv('query2_top_countries.csv', index=False)
pd.read_sql_query(query3, conn).to_csv('query3_top_products.csv', index=False)
pd.read_sql_query(query5, conn).to_csv('query5_monthly_sales.csv', index=False)

print("All CSVs exported successfully!")

All CSVs exported successfully!
